In [1]:
from kafka import KafkaConsumer, KafkaProducer
import json
import matplotlib.pyplot as plt
import time
import joblib
import numpy as np

In [2]:
# Initialize data storage for plotting
historial = {}
MODEL_PATH = "./aeration_classifier.joblib"
rf_model = joblib.load(MODEL_PATH)
PROCESADOR = "ML A"

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
# Kafka configuration
def initialize_consumer():
    # Kafka configuration
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]
    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
        )
    return consumer

def initialize_result_producer():
    kafka_bootstrap_servers = ["localhost:9092"]
    producer = KafkaProducer(bootstrap_servers=kafka_bootstrap_servers, value_serializer=lambda v: json.dumps(v).encode('utf-8'))
    return producer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer, producer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            sensor_id = message.key.decode("utf-8") if message.key else "no_key"
            print(f"Received: sensor_id:{sensor_id},{sensor_data}")

            predicciones = [[sensor_data['water_temperature'], sensor_data['ph_level'], sensor_data['turbidity'], sensor_data['dissolved_oxygen']]]
            
            proba = rf_model.predict_proba(predicciones)[0]   # [p(clase0), p(clase1)]
            pred = int(rf_model.predict(predicciones)[0])     # 0/1
            
            resultado = {
                "sensor_id": sensor_id,
                "timestamp": sensor_data['timestamp'],
                "p_class_0": float(proba[0]),
                "p_class_1": float(proba[1]),
                "pred": pred
            }
            if sensor_id not in list(historial.keys()):
                historial[sensor_id] = [[],[], [], []]
            historial[sensor_id][0].append(resultado['timestamp'])
            historial[sensor_id][1].append(resultado['p_class_0'])
            historial[sensor_id][2].append(resultado['p_class_1'])
            historial[sensor_id][3].append(resultado['pred'])
            if len(historial) > 100:
                historial[sensor_id][0].pop(0)
                historial[sensor_id][1].pop(0)
                historial[sensor_id][2].pop(0)
                historial[sensor_id][3].pop(0)
            
            topico = "water_end"
            producer.send(topico, key=message.key, value=resultado)
            print(f"Datos del proceso de aireación {PROCESADOR} del sensor {sensor_id}: {resultado}")
            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(historial[sensor_id][0], historial[sensor_id][3], label="Estado arieación por timestamp", color="blue")
            plt.title("Estado arieación por timestamp")
            plt.ylabel("Arieación")

            plt.subplot(2, 2, 2)
            plt.plot(historial[sensor_id][0], historial[sensor_id][1], label="Confianza clase 0/arieación", color="green")
            plt.title("Confianza clase 0/arieación")
            plt.ylabel("Probabilidad clase 0")

            plt.subplot(2, 2, 3)
            plt.plot(historial[sensor_id][0], historial[sensor_id][2], label="Confianza clase 1/no arieación", color="yellow")
            plt.title("Confianza clase 1/arieación")
            plt.ylabel("Probabilidad clase 1")

            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(f"water_quality_plot{sensor_id}.png")
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [ ]:
consumer = initialize_consumer()
producer = initialize_result_producer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer, producer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: sensor_id:0,{'timestamp': 1772387426, 'water_temperature': 30.19614578637826, 'ph_level': 8.524164701649378, 'turbidity': 32.04, 'dissolved_oxygen': 10.63}
Datos del proceso de aireación ML A del sensor 0: {'sensor_id': '0', 'timestamp': 1772387426, 'p_class_0': 1.0, 'p_class_1': 0.0, 'pred': 0}
Received: sensor_id:1,{'timestamp': 1772387426, 'water_temperature': 29.62177254853642, 'ph_level': 8.364979193675946, 'turbidity': 38.45, 'dissolved_oxygen': 10.76}
Datos del proceso de aireación ML A del sensor 1: {'sensor_id': '1', 'timestamp': 1772387426, 'p_class_0': 1.0, 'p_class_1': 0.0, 'pred': 0}
Received: sensor_id:4,{'timestamp': 1772387426, 'water_temperature': 28.801746926912973, 'ph_level': 8.482991270789848, 'turbidity': 23.79, 'dissolved_oxygen': 11.03}
Datos del proceso de aireación ML A del sensor 4: {'sensor_id': '4', 'timestamp': 1772387426, 'p_class_0': 1.0, 'p_class_1': 0.0, 'pred': 0}
Received: sensor_id:3,{'timestamp':